In [1]:
# Import Library
import pandas as pd
import os
import re

## Noise Removal

In [2]:
# 1.1 Load Data Hasil Deduplication
df = pd.read_csv('../../outputs/data-preparation/dataset_selected_no_stem.csv')

print("Data berhasil dimuat.")
print("Jumlah data sebelum noise removal:", len(df))

df.head()

Data berhasil dimuat.
Jumlah data sebelum noise removal: 16205


,timestamp,text
0,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik & negar...
1,2016-12-30T06:30:36.000Z,"Tertibkan Media Online, DPR: Pemerintah Jangan..."
2,2016-12-30T04:48:35.000Z,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...
3,2016-12-30T04:21:40.000Z,"jangan ngambang, aturan logis apa undang-undan..."
4,2016-12-30T02:36:13.000Z,6. Kebebasan bersuara & berpendapat memang dij...


In [3]:
# 1.2 Definisi Fungsi Noise Removal

def noise_removal(text):
    text = str(text)

    text = re.sub(r"http\S+|www\S+", "", text)   # hapus URL
    text = re.sub(r"@\w+", "", text)             # hapus mention
    text = re.sub(r"#(\w+)", r"\1", text)        # hapus simbol # tapi simpan katanya
    text = re.sub(r"&\w+;", "", text)            # hapus HTML entity
    text = re.sub(r"\d+", "", text)              # hapus angka
    text = re.sub(r"[^\w\s!?]", " ", text)       # hapus tanda baca kecuali ! dan ?
    text = re.sub(r"\s+", " ", text).strip()     # rapikan spasi

    return text

In [4]:
# 1.3 Menerapkan Noise Removal
df['text_clean'] = df['text'].apply(noise_removal)

df[['text', 'text_clean']].head()

,text,text_clean
0,ADIL loh utk yg punya kebijakan publik & negar...,ADIL loh utk yg punya kebijakan publik negara ...
1,"Tertibkan Media Online, DPR: Pemerintah Jangan...",Tertibkan Media Online DPR Pemerintah Jangan S...
2,@Portal_Kemlu_RI @DPR_RI @jokowi harus dievalu...,harus dievaluasi lg kebijakan bebas visa truta...
3,"jangan ngambang, aturan logis apa undang-undan...",jangan ngambang aturan logis apa undang undang
4,6. Kebebasan bersuara & berpendapat memang dij...,Kebebasan bersuara berpendapat memang dijamin ...


In [5]:
# Mengatur display pandas agar lebih rapi
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.max_colwidth', 100)  # Batasi lebar kolom saat preview

# Fungsi helper untuk memotong teks panjang 
def truncate_text(text, max_len=80):
    """Memotong teks menjadi max_len karakter + '...' jika terlalu panjang"""
    if pd.isna(text):
        return ""
    text = str(text)
    return text[:max_len] + "..." if len(text) > max_len else text

# 1.4 Filter Karakter Non-Latin
non_latin_mask = df['text_clean'].str.contains(
    r'[\uAC00-\uD7A3\u1100-\u11FF\u3130-\u318F'  # Hangul (Korea)
    r'\u4E00-\u9FFF\u3400-\u4DBF'                # CJK Unified dan Ext-A (Kanji/Hanzi)
    r'\u0400-\u04FF'                             # Kiril
    r'\u0370-\u03FF'                             # Yunani
    r'\u0E00-\u0E7F\u0900-\u097F]'               # Thai dan Devanagari
    , regex=True, na=False
)

count_non_latin = non_latin_mask.sum()
print(f"[VALIDASI] Ditemukan {count_non_latin} baris mengandung tulisan non-Latin.\n")

if count_non_latin > 0:
    print("[PREVIEW] Contoh data non-Latin yang akan dihapus:")
    print("-" * 120)
    
    # Ambil sampel, lalu buat kolom preview yang sudah dipotong
    preview_df = df.loc[non_latin_mask, ['text', 'text_clean']].head(3).copy()
    preview_df['text_preview'] = preview_df['text'].apply(truncate_text, max_len=70)
    preview_df['clean_preview'] = preview_df['text_clean'].apply(truncate_text, max_len=70)
    
    # Tampilkan hanya kolom preview yang sudah dipotong
    print(preview_df[['text_preview', 'clean_preview']].to_string(index=False))
    print("-" * 120 + "\n")
    
    # Menunjukkan karakter non-Latin apa yang terdeteksi
    import re
    print("[DETAIL] Karakter non-Latin yang terdeteksi:")
    for idx, row in df.loc[non_latin_mask, ['text_clean']].head(3).iterrows():
        non_latin_chars = re.findall(r'[\uAC00-\uD7A3\u4E00-\u9FFF\u0600-\u06FF\u0400-\u04FF]', str(row['text_clean']))
        if non_latin_chars:
            print(f"  • Baris {idx}: {list(set(non_latin_chars))}")
    print()
    
    # Eksekusi penghapusan
    df = df[~non_latin_mask].reset_index(drop=True)
    print(f"[INFO] {count_non_latin} baris berhasil dihapus.")

print(f"[STATUS] Jumlah data setelah filter non-Latin: {len(df):,}")

[VALIDASI] Ditemukan 1 baris mengandung tulisan non-Latin.

[PREVIEW] Contoh data non-Latin yang akan dihapus:
------------------------------------------------------------------------------------------------------------------------
                                                         text_preview                                                         clean_preview
국회 진입했던 계엄군들 철수 시작 계엄령 해제 국회의장 국회 진입 전원 찬성 계엄사령관 비상계엄 우리나라 공수부대 국회 본청 국회 진입했던 계엄군들 철수 시작 계엄령 해제 국회의장 국회 진입 전원 찬성 계엄사령관 비상계엄 우리나라 공수부대 국회 본청
------------------------------------------------------------------------------------------------------------------------

[DETAIL] Karakter non-Latin yang terdeteksi:
  • Baris 15932: ['엄', '본', '철', '우', '원', '관', '계', '청', '령', '던', '비', '들', '나', '했', '입', '수', '해', '라', '리', '군', '부', '장', '제', '전', '회', '상', '진', '대', '찬', '작', '국', '시', '성', '의', '사', '공']

[INFO] 1 baris berhasil dihapus.
[STATUS] Jumlah data setelah filter non-Latin: 16,204


In [6]:
# 1.5 Cek dan Hapus Teks Kosong Setelah Noise Removal dan Filter tulisan non-latin
kosong_setelah_noise = df['text_clean'].str.strip().eq('').sum()

print("\nJumlah text_clean kosong setelah noise removal:", kosong_setelah_noise)

if kosong_setelah_noise > 0:
    df = df[df['text_clean'].str.strip() != '']
    print(f"Jumlah data setelah hapus text_clean kosong: {len(df)}")


Jumlah text_clean kosong setelah noise removal: 22
Jumlah data setelah hapus text_clean kosong: 16182


In [7]:
# 1.6 Simpan Checkpoint Setelah Noise Removal

output_path = '../../outputs/data-preparation'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_noise_removal_no_stem.csv'),
    index=False
)

print("Checkpoint data_after_noise_removal_no_stem.csv berhasil disimpan.")

Checkpoint data_after_noise_removal_no_stem.csv berhasil disimpan.


### CHECKING MISSING VALUES

In [8]:
# Base path configuration
BASE_PATH = '../../'

# 2.1 Load Data Hasil Noise Removal
df = pd.read_csv(os.path.join(BASE_PATH, 'outputs/data-preparation/data_after_noise_removal_no_stem.csv'))

print("Data hasil noise removal berhasil dimuat.")

Data hasil noise removal berhasil dimuat.


In [9]:
# 2.2 Ringkasan Missing Values (NaN)

missing_before = df.isnull().sum()

print("Jumlah missing value sebelum cleaning:")
print(missing_before)

Jumlah missing value sebelum cleaning:
timestamp     0
text          0
text_clean    0
dtype: int64


In [10]:
# 2.3 Checking Empty String / Blank Text

# Menghitung text kosong atau hanya berisi spasi
empty_text_count = df['text_clean'].astype(str).str.strip().eq('').sum()

print("Jumlah text kosong (string kosong / hanya spasi):", empty_text_count)

Jumlah text kosong (string kosong / hanya spasi): 0


In [11]:
# 2.4 Simpan Checkpoint Setelah Missing Values

output_path = '../../outputs/data-preparation'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_missing_values_no_stem.csv'),
    index=False
)

print("Checkpoint data_after_missing_values_no_stem.csv berhasil disimpan.")

Checkpoint data_after_missing_values_no_stem.csv berhasil disimpan.


### DATA DEDUPLICATION

In [12]:
# Base path configuration
BASE_PATH = '../../'

# 3.1 Load Data Hasil Missing Values
df = pd.read_csv(os.path.join(BASE_PATH, 'outputs/data-preparation/data_after_missing_values_no_stem.csv'))

print("Data hasil missing values berhasil dimuat.")

Data hasil missing values berhasil dimuat.


In [13]:
# 3.2 Ringkasan Duplikasi (Before Deduplication)

jumlah_data_sebelum = len(df)

# Menghitung jumlah baris yang terdeteksi sebagai duplikat
jumlah_duplikat = df.duplicated(subset=['text_clean']).sum()

print("Jumlah data sebelum deduplication:", jumlah_data_sebelum)
print("Jumlah data duplikat terdeteksi:", jumlah_duplikat)

Jumlah data sebelum deduplication: 16182
Jumlah data duplikat terdeteksi: 2990


In [14]:
# 3.3 Menampilkan Contoh Data Duplikat

df_duplikat_before = df[
    df.duplicated(subset=['text_clean'], keep=False)
]

print("Total baris yang termasuk kelompok duplikat:", len(df_duplikat_before))
df_duplikat_before[['text_clean']].head(10)

Total baris yang termasuk kelompok duplikat: 3987


,text_clean
24,Demikian ulasan singkat tentang Masa Reses DPR semoga bermanfaat bagi kita semua ResesDPR
25,Masa Reses ini diharapkan dapat dimanfaatkan dengan maksimal oleh masyarakat untuk menyampaikan ...
26,Yg perlu diingat kita pahami bersama bahwa DPR bukanlah eksekutor jadi temuan selama Masa Reses ...
27,Berdasarkan Tatib DPR Pasal ayat hasil Reses tersebut dilaporkan secara tertulis oleh Anggota ke...
28,Kegiatan reses para Anggota DPR dilakukan secara transparan dan adanya pertanggungjawaban yang j...
29,Pada Masa Persidangan aspirasi masukan masyarakat selama Masa Reses akan disampaikan dibahas dg ...
31,Masa Reses sendiri tertuang dalam UU MD No dan Peraturan DPR No tentang Tata Tertib DPR ResesDPR
32,Masa Reses bukan berarti para Anggota DPR tidak bekerja namuan aktivitasnya dilakukan di daerah ...
33,Dalam Masa Reses Anggota DPR akan melakukan kunjungan ke Dapil maupun kunjungan ke daerah lainny...
34,Masa Reses adalah masa dimana Anggota DPR melakukan kegiatan di luar Masa Sidang terutama di lua...


In [15]:
# 3.4 Pendetailan teks yang duplikat serta jumlah kemunculanya
import os

folder_path = "validasidata"
os.makedirs(folder_path, exist_ok=True)

# Hitung jumlah kemunculan tiap teks
duplicate_counts = df['text_clean'].value_counts()

# Ambil hanya yang duplikat (>1)
duplicate_counts = duplicate_counts[duplicate_counts > 1]

# Ubah ke DataFrame
duplicate_detail = duplicate_counts.reset_index()
duplicate_detail.columns = ['text_clean', 'jumlah_kemunculan']

# Preview 10 data teratas
print("Contoh 10 data duplikat beserta jumlah kemunculannya:")
print(duplicate_detail.head(10))

# Simpan ke CSV
file_path = os.path.join(folder_path, "detailduplikat2.csv")
duplicate_detail.to_csv(file_path, index=False)

print(f"\nData lengkap berhasil disimpan di: {file_path}")

Contoh 10 data duplikat beserta jumlah kemunculannya:
                                                                                            text_clean  jumlah_kemunculan
0                                                    Henry Yosodiningrat Persen Anggota DPR Ikut Reses                 68
1               Di Ngawi Komisi X DPR RI Siapakan Delapan Item RUU Kebudayaan siaga indonesia Sindiran                 39
2                             DPR RI Bersama Kemenpar RI Selenggarakan Bimtek Kebijakan Promosi Wisata                 31
3                                                               Anggota DPR RUU ITE mengacu putusan MK                 25
4                                                        Anggota DPR ada pasal krusia dalam RUU Pemilu                 24
5                                                    Anggota DPR pertanyakan dana repatriasi dalam RUU                 23
6                                                   Anggota DPR Ini Minta Kebijakan Bebas Vi

In [16]:
# 3.5 Proses Deduplication

df = df.drop_duplicates(
    subset=['text_clean'],
    keep='first'
)

print("Proses deduplication selesai.")

Proses deduplication selesai.


In [17]:
# 3.6 Perbandingan Jumlah Data Before–After

jumlah_data_setelah = len(df)
jumlah_data_dihapus = jumlah_data_sebelum - jumlah_data_setelah

print("Jumlah data setelah deduplication:", jumlah_data_setelah)
print("Jumlah data yang dihapus:", jumlah_data_dihapus)

Jumlah data setelah deduplication: 13192
Jumlah data yang dihapus: 2990


In [18]:
# 3.7 Verifikasi Duplikasi Setelah Deduplication

duplikat_setelah = df.duplicated(subset=['text_clean']).sum()

print("Jumlah duplikat setelah deduplication:", duplikat_setelah)

Jumlah duplikat setelah deduplication: 0


In [19]:
# 3.8 Reset Index Setelah Deduplication

df = df.reset_index(drop=True)

print("Index berhasil direset.")

Index berhasil direset.


In [20]:
# 3.9 Simpan Checkpoint Data Setelah Deduplication

output_path = '../../outputs/data-preparation'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_after_deduplication_no_stem.csv'),
    index=False
)

print("Checkpoint data_after_deduplication_no_stem.csv berhasil disimpan.")

Checkpoint data_after_deduplication_no_stem.csv berhasil disimpan.


# FORMATING DAN PENGECEKKAN ULANG DATA

In [21]:
# Base path configuration
BASE_PATH = '../../'

# 4.1 Load Data After Deduplication
df = pd.read_csv(os.path.join(BASE_PATH, 'outputs/data-preparation/data_after_deduplication_no_stem.csv'))

print("Jumlah data:", len(df))

Jumlah data: 13192


In [22]:
# 4.2 Formatting Final Dataset

# Rename kolom text_clean menjadi teks
df = df.rename(columns={'text_clean': 'teks'})

# Hapus kolom text lama
df = df.drop(columns=['text'], errors='ignore')

# Reset index dulu
df = df.reset_index(drop=True)

# Tambahkan nomor urut
df.insert(0, 'no', range(1, len(df) + 1))

# Atur ulang kolom
df = df[['no', 'timestamp', 'teks']]

print("Struktur akhir dataset:")
df.head()

Struktur akhir dataset:


,no,timestamp,teks
0,1,2016-12-30T06:37:56.000Z,ADIL loh utk yg punya kebijakan publik negara Ingat yg ini!!!
1,2,2016-12-30T06:30:36.000Z,Tertibkan Media Online DPR Pemerintah Jangan Sporadis Apalagi Selektif Hanya kepada Media yang B...
2,3,2016-12-30T04:48:35.000Z,harus dievaluasi lg kebijakan bebas visa trutama utk negara tiongkok pak!! bahaya tersembunyi ma...
3,4,2016-12-30T04:21:40.000Z,jangan ngambang aturan logis apa undang undang
4,5,2016-12-30T02:36:13.000Z,Kebebasan bersuara berpendapat memang dijamin UU Namun kebebasan tersebut tdk harus kebablasan s...


In [23]:
# 4.3 Simpan Dataset Final Cleaning

output_path = '../../outputs/data-preparation'
os.makedirs(output_path, exist_ok=True)

df.to_csv(
    os.path.join(output_path, 'data_cleaning_final_no_stem.csv'),
    index=False
)

print("data_cleaning_final_no_stem.csv berhasil disimpan.")

data_cleaning_final_no_stem.csv berhasil disimpan.


In [24]:
print("Duplikat:", df.duplicated(subset=['teks']).sum())
print("NaN:", df['teks'].isna().sum())
print("Kosong:", (df['teks'].str.strip() == "").sum())
print("Total final:", len(df))

Duplikat: 0
NaN: 0
Kosong: 0
Total final: 13192
